# Fine-tune MusicGen (Hugging Face Transformers)

This notebook fine-tunes `facebook/musicgen-small` (swap in `-medium` if you have the VRAM) on your own audio + text captions.

**How it works**
1. Audio is cut into fixed-length chunks and tokenised once with the frozen EnCodec audio encoder.
2. The frozen T5 text encoder conditions the MusicGen decoder (the part we train).
3. Targets use MusicGen's *delay pattern* across the 4 codebooks, with cross-entropy loss per codebook.

**Your data:** put audio files in a folder (default `data/`). Captions come from, in priority order:
- a `.txt` file with the same name as the audio file (`song.wav` → `song.txt`)
- a `metadata.csv` in the folder with columns `file_name,caption`
- otherwise `DEFAULT_CAPTION`

**Hardware:** a free Colab T4 (16 GB) works for `musicgen-small` with the defaults. Use a GPU runtime.

In [ ]:
!pip install -q -U transformers accelerate librosa soundfile scipy

## 1. Config

In [ ]:
MODEL_NAME       = "facebook/musicgen-small"
DATA_DIR         = "data"            # folder with your audio (+ captions)
DEFAULT_CAPTION  = "lo-fi hip hop beat with warm piano"   # used if a file has no caption
OUTPUT_DIR       = "musicgen-finetuned"

CHUNK_SECONDS    = 10                # training clip length (MusicGen was trained on up to 30s)
MIN_CHUNK_SECONDS = 4                # drop leftover tail chunks shorter than this

BATCH_SIZE       = 4
GRAD_ACCUM       = 4                 # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS           = 10
LR               = 2e-5              # full fine-tune: keep this low (1e-5 to 5e-5)
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 20
MAX_GRAD_NORM    = 1.0
TEXT_DROPOUT     = 0.1               # randomly blank captions so classifier-free guidance still works
SEED             = 42

In [ ]:
import os, glob, csv, math, random
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from tqdm.auto import tqdm
from transformers import AutoProcessor, MusicgenForConditionalGeneration, get_cosine_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
assert device == "cuda", "Please use a GPU runtime (Runtime > Change runtime type)."

## 2. Load model

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

SR        = model.config.audio_encoder.sampling_rate     # 32000
FRAME_RATE = SR / model.config.audio_encoder.hop_length if hasattr(model.config.audio_encoder, "hop_length") else 50
K         = model.decoder.config.num_codebooks            # 4
PAD_ID    = model.decoder.config.pad_token_id             # 2048 (also used as BOS)
print(f"sampling rate={SR}, codebooks={K}, pad/bos id={PAD_ID}")

# Freeze the audio tokeniser and the text encoder; train decoder + text->decoder projection
for p in model.audio_encoder.parameters(): p.requires_grad = False
for p in model.text_encoder.parameters():  p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable/1e6:.1f}M")

## 3. Load audio + captions, chunk, and tokenise with EnCodec

In [ ]:
AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a")
files = sorted(f for f in glob.glob(f"{DATA_DIR}/**/*", recursive=True) if f.lower().endswith(AUDIO_EXTS))
assert files, f"No audio found in '{DATA_DIR}/'. Upload some files first."

# optional metadata.csv
meta = {}
meta_path = os.path.join(DATA_DIR, "metadata.csv")
if os.path.exists(meta_path):
    with open(meta_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            meta[row["file_name"]] = row["caption"]

def get_caption(path):
    txt = os.path.splitext(path)[0] + ".txt"
    if os.path.exists(txt):
        return open(txt, encoding="utf-8").read().strip()
    rel = os.path.relpath(path, DATA_DIR)
    if rel in meta: return meta[rel]
    if os.path.basename(path) in meta: return meta[os.path.basename(path)]
    return DEFAULT_CAPTION

chunk_len, min_len = CHUNK_SECONDS * SR, MIN_CHUNK_SECONDS * SR
chunks = []   # (waveform, caption)
for f in tqdm(files, desc="Loading audio"):
    wav, _ = librosa.load(f, sr=SR, mono=True)
    cap = get_caption(f)
    for start in range(0, len(wav), chunk_len):
        seg = wav[start:start + chunk_len]
        if len(seg) < min_len: continue
        if len(seg) < chunk_len: seg = np.pad(seg, (0, chunk_len - len(seg)))
        chunks.append((seg.astype(np.float32), cap))
print(f"{len(files)} files -> {len(chunks)} chunks of {CHUNK_SECONDS}s")
assert len(chunks) > 0

In [ ]:
# Encode all chunks to discrete audio tokens once (saves time every epoch)
model.audio_encoder.eval()
codes_list = []
with torch.no_grad():
    for i in tqdm(range(0, len(chunks), 8), desc="EnCodec"):
        wav = torch.from_numpy(np.stack([c[0] for c in chunks[i:i+8]])).unsqueeze(1).to(device)  # (B,1,N)
        out = model.audio_encoder.encode(wav)
        codes_list.append(out.audio_codes[0].cpu())            # (B, K, T)
all_codes = torch.cat(codes_list).long()                        # (N, K, T)
captions  = [c[1] for c in chunks]
print("codes:", tuple(all_codes.shape), "| example caption:", captions[0])
del chunks

In [ ]:
class CodeDataset(torch.utils.data.Dataset):
    def __len__(self): return len(all_codes)
    def __getitem__(self, i): return all_codes[i], captions[i]

def collate(batch):
    codes = torch.stack([b[0] for b in batch])
    texts = [("" if random.random() < TEXT_DROPOUT else b[1]) for b in batch]
    tok = processor.tokenizer(texts, padding=True, return_tensors="pt")
    return codes, tok["input_ids"], tok["attention_mask"]

loader = torch.utils.data.DataLoader(CodeDataset(), batch_size=BATCH_SIZE, shuffle=True,
                                     collate_fn=collate, drop_last=len(all_codes) >= BATCH_SIZE)
print("batches per epoch:", len(loader))

## 4. Delay-pattern targets + sanity check

MusicGen predicts the 4 codebooks with a one-step delay between them. We build that pattern with the model's own helper, then do standard teacher forcing (inputs = sequence[:-1], labels = sequence[1:]). Padding/BOS positions are ignored in the loss.

In [ ]:
def make_inputs_and_labels(codes):
    """codes: (B, K, T) -> decoder_input_ids (B*K, L-1), labels (B*K, L-1)"""
    B, _, T = codes.shape
    flat = codes.reshape(B * K, T)
    _, seq = model.decoder.build_delay_pattern_mask(flat, PAD_ID, max_length=T + K)
    seq = seq.masked_fill(seq == -1, PAD_ID)          # any unfilled slots -> ignored
    dec_in = seq[:, :-1]
    labels = seq[:, 1:].masked_fill(seq[:, 1:] == PAD_ID, -100)
    return dec_in, labels

# sanity check on one batch
codes, ids, am = next(iter(loader))
dec_in, labels = make_inputs_and_labels(codes.to(device))
print("decoder_input_ids:", tuple(dec_in.shape), "labels:", tuple(labels.shape))
with torch.no_grad():
    out = model(input_ids=ids.to(device), attention_mask=am.to(device), decoder_input_ids=dec_in)
print("logits:", tuple(out.logits.shape))
assert out.logits.shape[:2] == dec_in.shape, "Unexpected logits shape - check your transformers version"

## 5. Train

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=not use_bf16)
print(f"AMP dtype: {amp_dtype}, optimizer steps: {total_steps}")

history = []
for epoch in range(EPOCHS):
    model.train()
    model.text_encoder.eval(); model.audio_encoder.eval()
    running, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    bar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for i, (codes, ids, am) in enumerate(bar):
        codes, ids, am = codes.to(device), ids.to(device), am.to(device)
        dec_in, labels = make_inputs_and_labels(codes)

        with torch.autocast("cuda", dtype=amp_dtype):
            out = model(input_ids=ids, attention_mask=am, decoder_input_ids=dec_in)
        logits = out.logits.float()
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)

        scaler.scale(loss / GRAD_ACCUM).backward()
        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running += loss.item(); n += 1
        bar.set_postfix(loss=f"{running/n:.4f}")
    history.append(running / n)
    print(f"Epoch {epoch+1}: loss {history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(1, len(history)+1), history, marker="o")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.grid(alpha=.3); plt.show()

## 6. Generate with the fine-tuned model

In [ ]:
from IPython.display import Audio, display
import scipy.io.wavfile as wavfile

model.eval()
prompts = [
    DEFAULT_CAPTION,
    "mellow acoustic guitar with soft percussion",
]
SECONDS = 10
inputs = processor(text=prompts, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    audio = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(SECONDS * 50))

os.makedirs("samples", exist_ok=True)
for p, a in zip(prompts, audio):
    a = a[0].float().cpu().numpy()
    print(p)
    display(Audio(a, rate=SR))
    wavfile.write(f"samples/{p[:40].replace(' ', '_')}.wav", SR, a)

## 7. Save (and optionally push to the Hub)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

# Optional: push to the Hugging Face Hub
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/musicgen-small-finetuned")
# processor.push_to_hub("your-username/musicgen-small-finetuned")

# Optional (Colab): download as zip
# !zip -r musicgen-finetuned.zip musicgen-finetuned
# from google.colab import files; files.download("musicgen-finetuned.zip")

## Use it later
```python
from transformers import AutoProcessor, MusicgenForConditionalGeneration
processor = AutoProcessor.from_pretrained("musicgen-finetuned")
model = MusicgenForConditionalGeneration.from_pretrained("musicgen-finetuned").to("cuda")
```

## Tips
- **Out of memory?** Lower `BATCH_SIZE` (raise `GRAD_ACCUM` to compensate) or `CHUNK_SECONDS`.
- **Overfitting / losing general ability?** Use fewer epochs, lower `LR`, or add more varied data. With tiny datasets (< 30 min), 5-10 epochs is usually plenty.
- **Better captions = better control.** Describe genre, instruments, mood, tempo. Use consistent wording, and a unique trigger phrase if you want to "teach" a style.
- **Bigger models:** `facebook/musicgen-medium` needs ~24 GB+ for full fine-tuning; consider freezing layers or using LoRA via `peft`.
- Only fine-tune on audio you have the rights to use.